# ALFRED 半导体生产者价格历史版本来源核查

仅按冻结 `plan.json` 下载 BLS 半导体制造业 PPI 的 93 个 ALFRED 日期版本；不读 SOXX 收益，不训练模型。首次版本与后续修订必须分开。


In [ ]:
from pathlib import Path
import hashlib, io, json, zipfile
import requests
from bs4 import BeautifulSoup
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root: raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/semiconductor_ppi_alfred_source_20261022"
plan=json.loads((card/"plan.json").read_text())
assert plan["frozen"] and plan["execution_authorized"]
assert not (card/"retrieval_receipt.json").exists()
sha=lambda b:hashlib.sha256(b).hexdigest()
session=requests.Session()
response=session.get(plan["source_url"],timeout=30)
response.raise_for_status()
soup=BeautifulSoup(response.content,"html.parser")
form=next(f for f in soup.find_all("form") if f.find(attrs={"name":"form[file_type]"}) is not None)
choices=[x.get("value") for x in form.find(attrs={"name":"form[selected_vintage_dates][]"}).find_all("option")]
dates=[d for d in choices if plan["vintage_start"]<=d<=plan["vintage_end"]]
assert len(dates)==plan["expected_selected_vintages"]
assert dates[0]==plan["preflight_form_observation"]["first_selected"]
assert dates[-1]==plan["preflight_form_observation"]["last_selected"]
assert form.find(attrs={"name":"form[obs_end_date]"}).get("value")>=plan["observation_end"]
data=[("form[units]","lin"),("form[obs_start_date]",plan["observation_start"]),
    ("form[obs_end_date]",plan["observation_end"]),("form[entered_vintage_dates]",""),
    ("form[file_type]",plan["download_file_type"]),("form[file_format]",plan["download_file_format"]),
    ("form[download_data]","")]
data += [("form[selected_vintage_dates][]",d) for d in dates]
data += [(h.get("name"),h.get("value","")) for h in form.find_all("input",attrs={"type":"hidden"})]
result=session.post(plan["source_url"],data=data,timeout=60)
result.raise_for_status()
body=result.content
assert len(body)<=plan["request_budget"]["max_archive_bytes"]
assert zipfile.is_zipfile(io.BytesIO(body))
raw=root/plan["raw_path"]
raw.parent.mkdir(parents=True,exist_ok=True)
assert not raw.exists()
raw.write_bytes(body)
archive=zipfile.ZipFile(io.BytesIO(body))
names=archive.namelist()
receipt={"schema_version":1,"status":"source_archive_downloaded_schema_qualification_pending",
    "plan_sha256":sha((card/"plan.json").read_bytes()),"source_url":plan["source_url"],
    "selected_vintage_dates":len(dates),"first_vintage":dates[0],"last_vintage":dates[-1],
    "form_sha256":sha(response.content),"raw_path":plan["raw_path"],"raw_sha256":sha(body),
    "raw_bytes":len(body),"archive_entries":names,"market_outcome_reads":0,"model_fits":0,"accounts":0}
(card/"retrieval_receipt.json").write_text(json.dumps(receipt,indent=2)+"\n")
print(json.dumps({"status":receipt["status"],"vintages":len(dates),"archive_entries":names,"raw_bytes":len(body)}))


## 中文结论

执行后核对 `retrieval_receipt.json` 与原始 ZIP，随后另做同版本数值、缺失与发布时序审查。当前没有价格因子或组合结论。
